In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import warnings
from tqdm import tqdm
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

In [ ]:
class CFG:
    IMG_SIZE       = 224
    BATCH_SIZE     = 64
    EPOCHS         = 25
    LR             = 3e-4
    NUM_CLASSES    = 8
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
    EARLY_STOPPING = 7
    SEED           = 42
    DATA_PATH      = "/kaggle/input/competitions/icdar-2026-circleid-pen-classification/"
    MODEL_NAMES = [
                    "convnext_small.fb_in22k_ft_in1k",
                    "swin_small_patch4_window7_224.ms_in22k_ft_in1k",
                    "tf_efficientnetv2_s.in21k_ft_in1k",
                    "resnext101_32x4d.fb_swsl_ig1b_ft_in1k"
                    ]
    MODEL_WEIGHTS = {
                    "convnext_small.fb_in22k_ft_in1k": 0.35,
                    "swin_small_patch4_window7_224.ms_in22k_ft_in1k": 0.35,
                    "tf_efficientnetv2_s.in21k_ft_in1k": 0.2,
                    "resnext101_32x4d.fb_swsl_ig1b_ft_in1k": 0.1
}
    LABEL_SMOOTH   = 0.1
    TTA_STEPS      = 20
    MIXUP_ALPHA    = 0.3
    CUTMIX_ALPHA   = 1.0

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

seed_everything(CFG.SEED)

In [ ]:
train = pd.read_csv(CFG.DATA_PATH + 'train.csv')
extra = pd.read_csv(CFG.DATA_PATH + 'additional_train.csv')
test  = pd.read_csv(CFG.DATA_PATH + 'test.csv')

# Filter unknown pen_ids before concat
train = train[train['pen_id'] != -1].reset_index(drop=True)
extra = extra[extra['pen_id'] != -1].reset_index(drop=True)

train = pd.concat([train, extra], ignore_index=True)

# Verify
print(f"Total samples: {len(train)}")
print(f"Pen ID range: {train['pen_id'].min()} - {train['pen_id'].max()}")
print(f"Unique pen_ids: {sorted(train['pen_id'].unique())}")

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(CFG.IMG_SIZE, scale=(0.7, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.2)
])

val_transforms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

tta_transforms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

In [ ]:
class PenDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(CFG.DATA_PATH + row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(row['pen_id']) - 1  # 1-8 → 0-7
        return image, torch.tensor(label, dtype=torch.long)

class TestDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(CFG.DATA_PATH + row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, row['image_id']

In [ ]:
train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=CFG.SEED, stratify=train['pen_id']
)

train_loader    = DataLoader(PenDataset(train_df, train_transforms), batch_size=CFG.BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader      = DataLoader(PenDataset(val_df,   val_transforms),   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader     = DataLoader(TestDataset(test,    val_transforms),   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader_tta = DataLoader(TestDataset(test,    tta_transforms),   batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Verify labels
images, labels = next(iter(train_loader))
print(f"Label range: {labels.min()} - {labels.max()}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test)}")

In [ ]:
def mixup_cutmix(x, y, alpha=0.3):
    lam        = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index      = torch.randperm(batch_size).to(x.device)

    if np.random.rand() < 0.5:
        mixed_x = lam * x + (1 - lam) * x[index]
    else:
        _, _, H, W = x.shape
        cx    = np.random.randint(W)
        cy    = np.random.randint(H)
        cut_w = int(W * np.sqrt(1 - lam))
        cut_h = int(H * np.sqrt(1 - lam))
        x1    = np.clip(cx - cut_w // 2, 0, W)
        x2    = np.clip(cx + cut_w // 2, 0, W)
        y1    = np.clip(cy - cut_h // 2, 0, H)
        y2    = np.clip(cy + cut_h // 2, 0, H)
        mixed_x = x.clone()
        mixed_x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
        lam = 1 - ((x2-x1)*(y2-y1)/(W*H))

    return mixed_x, y, y[index], lam

In [ ]:
def build_model(model_name):
    model = timm.create_model(model_name, pretrained=True, num_classes=CFG.NUM_CLASSES)
    model = model.to(CFG.DEVICE)
    model = model.to(memory_format=torch.channels_last)
    return model

In [ ]:
def predict_with_tta(model_name):
    model_path = f"/kaggle/working/{model_name.replace('/', '_')}.pth"
    model      = build_model(model_name)
    checkpoint = torch.load(model_path, map_location=CFG.DEVICE)
    model.load_state_dict(checkpoint['model'])
    model.eval()

    n_test    = len(test)
    all_probs = np.zeros((n_test, CFG.NUM_CLASSES), dtype=np.float32)
    all_ids   = []

    with torch.no_grad():
        offset = 0
        for images, ids in tqdm(test_loader, desc=f"Inference TTA"):
            images = images.to(CFG.DEVICE)
            probs  = 0

            for _ in range(CFG.TTA_STEPS):
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                probs += torch.softmax(outputs, dim=1)

            probs = (probs / CFG.TTA_STEPS).cpu().numpy()
            all_probs[offset:offset+len(probs)] += probs
            offset += len(probs)
            all_ids.extend(ids)

    final_preds = np.argmax(all_probs, axis=1) + 1
    return all_probs, all_ids, final_preds

In [ ]:
def train_model(model_name):
    print(f"\nTraining {model_name}")

    model     = build_model(model_name)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, eta_min=1e-6)
    scaler    = torch.cuda.amp.GradScaler()

    best_acc = 0.0
    patience = 0

    for epoch in range(CFG.EPOCHS):
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.EPOCHS} [Train]"):
            images = images.to(CFG.DEVICE, memory_format=torch.channels_last)
            labels = labels.to(CFG.DEVICE)
            images, y_a, y_b, lam = mixup_cutmix(images, labels, CFG.MIXUP_ALPHA)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss    = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * images.size(0)
            correct    += (torch.argmax(outputs, dim=1) == labels).sum().item()
            total      += labels.size(0)

        scheduler.step(epoch)

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG.EPOCHS} [Val]  "):
                images, labels = images.to(CFG.DEVICE), labels.to(CFG.DEVICE)
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                val_correct += (torch.argmax(outputs, dim=1) == labels).sum().item()
                val_total   += labels.size(0)

        train_acc = correct     / total
        val_acc   = val_correct / val_total
        cur_lr    = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{CFG.EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {cur_lr:.2e}")

        if val_acc > best_acc:
            best_acc = val_acc
            patience = 0
            path = f"/kaggle/working/{model_name.replace('/', '_')}.pth"
            torch.save({'model': model.state_dict()}, path)
            print(f"  ✓ New best saved ({val_acc:.4f})")
        else:
            patience += 1
            print(f"  No improvement ({patience}/{CFG.EARLY_STOPPING})")
            if patience >= CFG.EARLY_STOPPING:
                print("Early stopping triggered")
                break

    print(f"\nBest Val Accuracy: {best_acc:.4f}")
    return best_acc

In [ ]:
for model_name in CFG.MODEL_NAMES:
    best_acc = train_model(model_name)
    print(f"{model_name} best val acc: {best_acc:.4f}")


In [ ]:
all_probs = None
all_ids   = None

for model_name in CFG.MODEL_NAMES:
    probs, ids, _ = predict_with_tta(model_name)
    
    if all_probs is None:
        all_probs = probs * CFG.MODEL_WEIGHTS[model_name]  # assign, not +=
        all_ids   = ids
    else:
        all_probs += probs * CFG.MODEL_WEIGHTS[model_name]

final_preds = np.argmax(all_probs, axis=1) + 1

pd.DataFrame({
    'image_id': all_ids,
    'pen_id':   final_preds
}).to_csv('/kaggle/working/submission.csv', index=False)
print("Ensemble submission saved!")